In [0]:
%run ./data_utils

In [0]:

# Get C table name from T table
def get_c_table_name(t_table_name):
    if '.' not in t_table_name:
        raise Exception('parameter invalid. missing catelog or schema. => ' + t_table_name)
    idx = t_table_name.rindex('.')
    tb_prefix = t_table_name[:idx]
    t_table = t_table_name[(idx+1):]
    if t_table.startswith('t_'):
        c_table = 'c_' + t_table[2:]
    else:
        c_table = 'c_' + t_table
    return tb_prefix + '.' + c_table 


def init_c_table(df, t_table_name, cdc_operation_time, cdc_operation_version):
    target_table_c = get_c_table_name(t_table_name)
    df = (df.withColumn("CDC_CODE", F.lit("I"))
            .withColumn("CDC_OperationTime", F.lit(cdc_operation_time))
            .withColumn("CDC_OperationVersion", F.lit(cdc_operation_version))
        )
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(target_table_c)


# Merge C table
def merge_c_table(t_table_name, version):
    print(f"ttable name: {t_table_name}")

    # find version
    vdf = spark.sql(f'''
                SELECT version, operation, timestamp
                FROM (DESCRIBE HISTORY {t_table_name}) 
                WHERE version = {version}
                '''
        ).cache()
    
    if vdf.isEmpty():
        raise Exception(f"table {t_table_name} version is not found: {version}" )

    version_info = vdf.select("version", "operation", "timestamp").collect()[0]

    # first merge
    if version_info[1] == "CREATE OR REPLACE TABLE AS SELECT":
        print(f'>> CDC init By CREATE OR REPLACE TABLE \n')
        init_c_table(spark.table(t_table_name), t_table_name, version_info[2], version_info[0])
        return

    # merge
    # get changed data
    cur_version = version_info[0]
    cur_version_timestamp = version_info[2]
    query = f"SELECT * FROM table_changes('{t_table_name}', {cur_version}, {cur_version}) WHERE _change_type <> 'update_preimage'"
    cdf = (spark.sql(query)
                .withColumn("CDC_CODE", 
                        F.when(F.col("_change_type")=="insert", F.lit("I"))
                        .when(F.col("_change_type")=="update_postimage", F.lit("U"))
                        .when(F.col("_change_type")=="delete", F.lit("D"))
                        .otherwise(F.lit(""))
                )
                .withColumn("CDC_OperationTime", F.lit(cur_version_timestamp))
                .withColumn("CDC_OperationVersion", F.lit(cur_version))
                .withColumn("_create_time", F.current_timestamp())
                .drop('_change_type', '_commit_version', '_commit_timestamp')
        )
    # append to c table
    target_table_c = get_c_table_name(t_table_name)
    print(f"ctable name: {target_table_c}")

    if not spark.catalog.tableExists(target_table_c):
        print(f'>> CDC init by ctable \n')
        cdf.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(target_table_c)
        return
    
    append_table(cdf, target_table_c)
    print(f'>> CDC merged successfully\n')

    vdf.unpersist()

In [0]:
def calc_ctable(t_table_name, t_table_version):

    if t_table_version == None:
        t_table_version = get_latest_version(t_table_name)

    print(f"merging ctable of {t_table_name}:{t_table_version}")
    merge_c_table(t_table_name, t_table_version)


In [0]:
# t_table_name = dbutils.widgets.get("t_table_name")
# t_table_version = get_ex_param("t_table_version", None)

# calc_ctable(t_table_name, t_table_version)